# scVI / scANVI Annotation — Cisplatin_GC_Vehicle (4 conditions)

**Author**: Xin Wang  
**Date**: 2026-04-14  

This notebook runs scVI/scANVI label transfer onto the 4-condition query dataset  
(Vehicle, Vehicle-GC, Cisplatin, Cisplatin-GC) using **two reference atlases**:

| Reference | File |
|-----------|------|
| Mouse Kidney Atlas (MKA) | `MouseKidneyATLAS_MKA_updated.h5ad` |
| Lake 2025 snRNA-seq | `mouse_kidney_snRNAseq_Lake2025_bioRxiv_V2.h5ad` |

**Pipeline:**
1. Load both reference h5ads; harmonise gene names and cell-type labels  
2. Load query h5ad (`Cisplatin_GC_Vehicle_Raw_Predict_Annotation.h5ad`)  
3. Merge references (batch-aware) and subset to shared genes with query  
4. Train scVI on merged reference  
5. Merge reference + query; train scANVI (semi-supervised)  
6. Predict cell types on query cells and save CSV  

**Prerequisites**: Run `Cisplatin_GC_Vehicle_Annotation.Rmd` first to generate the query h5ad.

In [ ]:
# Install dependencies if needed
# !pip install scanpy scvi-tools mygene

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import matplotlib.pyplot as plt
import torch

scvi.settings.seed = 0
sc.settings.verbosity = 1
print("scvi-tools version:", scvi.__version__)
print("GPU available:", torch.cuda.is_available())

## Paths

In [ ]:
ref_dir   = "/home/gdzepedaorozcolab/lab/xxw004/Projects/RawDZscRNAseq/Datasets/Reference/"
ann_dir   = "/home/gdzepedaorozcolab/lab/xxw004/Projects/RawDZscRNAseq/Results/Integration/IntegrationGC/Annotation/"

mka_path   = os.path.join(ref_dir, "MouseKidneyATLAS_MKA_updated.h5ad")
lake_path  = os.path.join(ref_dir, "mouse_kidney_snRNAseq_Lake2025_bioRxiv_V2.h5ad")
query_path = os.path.join(ann_dir, "Cisplatin_GC_Vehicle_Raw_Predict_Annotation.h5ad")
out_dir    = ann_dir
os.makedirs(out_dir, exist_ok=True)

for p in [mka_path, lake_path, query_path]:
    print(f"{os.path.basename(p):60s}  exists={os.path.exists(p)}")

## Load and inspect both reference datasets

In [ ]:
adata_mka  = sc.read_h5ad(mka_path)
adata_lake = sc.read_h5ad(lake_path)

print("=== MKA reference ===")
print(adata_mka)
print(adata_mka.obs.columns.tolist())

print("\n=== Lake reference ===")
print(adata_lake)
print(adata_lake.obs.columns.tolist())

In [ ]:
# -----------------------------------------------------------------------
# Identify the cell-type column in each reference.
# Update the values below if the column names differ in your h5ads.
# -----------------------------------------------------------------------
mka_label_col  = "cell_type"   # <-- update if needed
lake_label_col = "cell_type"   # <-- update if needed

print("MKA cell types:")
print(adata_mka.obs[mka_label_col].value_counts())

print("\nLake cell types:")
print(adata_lake.obs[lake_label_col].value_counts())

## Harmonise gene names (Ensembl → symbol if needed)

In [ ]:
def ensembl_to_symbol(adata):
    """Convert Ensembl IDs in var_names to gene symbols (mouse). No-op if already symbols."""
    if not adata.var_names[0].startswith("ENSMUS"):
        print("var_names already look like gene symbols — skipping conversion.")
        return adata
    import mygene
    mg = mygene.MyGeneInfo()
    ens_ids = list(adata.var_names)
    out = mg.querymany(ens_ids, scopes='ensembl.gene', fields='symbol', species='mouse', verbose=False)
    mapping = {d['query']: d.get('symbol', d['query']) for d in out}
    adata.var_names = [mapping.get(x, x) for x in adata.var_names]
    adata.var_names_make_unique()
    return adata

adata_mka  = ensembl_to_symbol(adata_mka)
adata_lake = ensembl_to_symbol(adata_lake)
print("MKA var sample:",  adata_mka.var_names[:5].tolist())
print("Lake var sample:", adata_lake.var_names[:5].tolist())

## Load query dataset (4-condition)

In [ ]:
adata_query = sc.read_h5ad(query_path)
print(adata_query)
print("Conditions:", adata_query.obs['DataSet'].value_counts().to_dict())

## Harmonise cell-type label column name and merge both references

In [ ]:
# Rename label columns to a unified name 'cell_type'
LABEL_KEY = "cell_type"

if mka_label_col != LABEL_KEY:
    adata_mka.obs[LABEL_KEY] = adata_mka.obs[mka_label_col]
if lake_label_col != LABEL_KEY:
    adata_lake.obs[LABEL_KEY] = adata_lake.obs[lake_label_col]

# Tag each reference with its source (used as the batch variable)
adata_mka.obs['ref_source']  = 'MKA'
adata_lake.obs['ref_source'] = 'Lake'

# Find genes shared across both references AND query
shared_genes = (
    adata_mka.var_names
    .intersection(adata_lake.var_names)
    .intersection(adata_query.var_names)
)
print(f"Shared genes (MKA ∩ Lake ∩ query): {len(shared_genes)}")
# Should be > 5000 for mouse

# Subset all three to shared genes
adata_mka_sub   = adata_mka[:, shared_genes].copy()
adata_lake_sub  = adata_lake[:, shared_genes].copy()
adata_query_sub = adata_query[:, shared_genes].copy()

In [ ]:
# Merge the two references
adata_ref_merged = adata_mka_sub.concatenate(
    adata_lake_sub,
    batch_key='ref_batch',
    batch_categories=['MKA', 'Lake']
)
# Keep cell_type from original obs (concatenate preserves columns with matching names)
print("Merged reference:", adata_ref_merged)
print("Cell type counts in merged reference:")
print(adata_ref_merged.obs[LABEL_KEY].value_counts())

## Train scVI on merged reference (batch-corrected)

In [ ]:
# scVI expects raw integer counts in .X
# If the data is log-normalised, pass layer='counts' (and ensure raw counts are stored there)
scvi.model.SCVI.setup_anndata(
    adata_ref_merged,
    labels_key='cell_type',
    batch_key='ref_batch'      # corrects for MKA vs Lake batch effects
)

scvi_ref = scvi.model.SCVI(adata_ref_merged)
scvi_ref.train()

scvi_ref_dir = os.path.join(out_dir, "scvi_model_dualref")
scvi_ref.save(scvi_ref_dir, overwrite=True)
print("scVI model saved to:", scvi_ref_dir)

In [ ]:
# Reference UMAP coloured by cell type and reference source
SCVI_LATENT_KEY = "X_scVI"
adata_ref_merged.obsm[SCVI_LATENT_KEY] = scvi_ref.get_latent_representation()
sc.pp.neighbors(adata_ref_merged, use_rep=SCVI_LATENT_KEY)
sc.tl.leiden(adata_ref_merged)
sc.tl.umap(adata_ref_merged)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sc.pl.umap(adata_ref_merged, color='cell_type',  title='Merged reference — cell type',   ax=axes[0], show=False)
sc.pl.umap(adata_ref_merged, color='ref_batch',  title='Merged reference — batch (MKA/Lake)', ax=axes[1], show=False)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'merged_reference_scVI_UMAP.pdf'), bbox_inches='tight')
plt.show()

## Prepare query for scANVI

In [ ]:
# Prepare query anndata to match the reference model's feature set
scvi.model.SCVI.prepare_query_anndata(adata_query_sub, scvi_ref)

# Mark query cells as 'Unknown' (unlabelled category for scANVI)
adata_query_sub.obs[LABEL_KEY] = pd.Categorical(['Unknown'] * adata_query_sub.n_obs)

# Also assign a ref_batch label for the query (keeps scANVI happy)
adata_query_sub.obs['ref_batch'] = 'query'

adata_query_sub.obs.head()

In [ ]:
# Merge reference + query for scANVI training
adata_combined = adata_ref_merged.concatenate(
    adata_query_sub,
    batch_key='batch',
    batch_categories=['ref', 'query']
)
print(adata_combined)
print("Batch distribution:")
print(adata_combined.obs['batch'].value_counts())

## Train scANVI (semi-supervised label propagation)

In [ ]:
scvi.model.SCANVI.setup_anndata(
    adata_combined,
    labels_key='cell_type',
    batch_key='batch',
    unlabeled_category='Unknown'
)

scanvi_model = scvi.model.SCANVI(adata_combined)
scanvi_model.train(max_epochs=50)

scanvi_dir = os.path.join(out_dir, "scanvi_model_dualref")
scanvi_model.save(scanvi_dir, overwrite=True)
print("scANVI model saved to:", scanvi_dir)

## Predict cell types on query cells

In [ ]:
query_mask = adata_combined.obs['batch'] == 'query'

query_preds = scanvi_model.predict(
    adata_combined[query_mask],
    soft=True
)

label_names = scanvi_model.adata_manager.get_state_registry('labels')[
    'categorical_mapping'
]
print("Labels in model:", label_names)

In [ ]:
# Build probability dataframe
prob_df = pd.DataFrame(
    query_preds,
    index=adata_combined.obs_names[query_mask],
    columns=label_names
)

pred_labels = prob_df.idxmax(axis=1)
pred_conf   = prob_df.max(axis=1)

adata_combined.obs.loc[query_mask, 'scanvi_label']      = pred_labels.values
adata_combined.obs.loc[query_mask, 'scanvi_confidence'] = pred_conf.values

adata_combined.obs.loc[query_mask, ['scanvi_label', 'scanvi_confidence']].head(10)

In [ ]:
# Confidence histogram
adata_combined.obs.loc[query_mask, 'scanvi_confidence'].hist(bins=50, figsize=(7, 4))
plt.title('scANVI Prediction Confidence — 4-condition query (dual reference)')
plt.xlabel('Confidence'); plt.ylabel('Cell count')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'scanvi_dualref_confidence_distribution.pdf'))
plt.show()

## Save predictions CSV (consumed by Annotation_Integration.Rmd)

In [ ]:
# Columns needed downstream in R
required_cols = [
    'predicted.id',            # from Seurat transfer (stored in h5ad obs)
    'prediction.score.max',    # from Seurat transfer
    'Raw_cell_type',           # from manual annotation
    'Raw_cell_type_Confidence',# from manual annotation (=1)
    'scanvi_label',            # scANVI prediction
    'scanvi_confidence'        # scANVI confidence
]

available = [c for c in required_cols if c in adata_combined.obs.columns]
missing   = [c for c in required_cols if c not in adata_combined.obs.columns]
if missing:
    print("WARNING — columns missing from obs (will be absent in CSV):", missing)

query_meta = adata_combined.obs.loc[query_mask, available].copy()
print(query_meta.shape)
query_meta.head()

In [ ]:
csv_path = os.path.join(out_dir, "Cisplatin_GC_Vehicle_scanvi_annotations.csv")
query_meta.to_csv(csv_path)
print("Predictions saved to:", csv_path)

# Save annotated query h5ad
adata_query_final = adata_combined[query_mask].copy()
adata_query_final.write(
    os.path.join(out_dir, "Cisplatin_GC_Vehicle_query_annotated_scanvi.h5ad")
)
print("Annotated query h5ad saved.")

## UMAP of combined reference + query

In [ ]:
adata_combined.obsm['X_scANVI'] = scanvi_model.get_latent_representation()
sc.pp.neighbors(adata_combined, use_rep='X_scANVI')
sc.tl.umap(adata_combined)

# Full combined UMAP
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
sc.pl.umap(adata_combined, color='cell_type',   title='scANVI cell type',  ax=axes[0], show=False)
sc.pl.umap(adata_combined, color='batch',       title='ref / query batch', ax=axes[1], show=False)
sc.pl.umap(adata_combined, color='ref_batch',   title='Reference source',  ax=axes[2], show=False)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'combined_scANVI_UMAP.pdf'), bbox_inches='tight')
plt.show()

# Query cells only, coloured by predicted label and condition
adata_q_vis = adata_combined[query_mask].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.umap(adata_q_vis, color='scanvi_label', title='Query — scANVI label',  ax=axes[0], show=False)
sc.pl.umap(adata_q_vis, color='DataSet',       title='Query — condition',     ax=axes[1], show=False)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'query_scANVI_UMAP.pdf'), bbox_inches='tight')
plt.show()

## Done

Next step: open **`Cisplatin_GC_Vehicle_Annotation_Integration.Rmd`** in R/RStudio  
to merge scANVI predictions with Seurat transfer and manual annotations  
into a final weighted consensus cell-type label.

Output CSV: `Cisplatin_GC_Vehicle_scanvi_annotations.csv`